# P1 LoRA 微调 — Qwen2.5-1.5B-Instruct + 伤寒论

本 notebook 在 Google Colab T4 GPU 上运行，完成：
1. 安装依赖
2. 加载 SFT 训练数据（1179 条伤寒论指令对）
3. LoRA 微调 Qwen2.5-1.5B-Instruct
4. 合并 adapter 到基座模型
5. 转为 GGUF + Q4_K_M 量化
6. 生成 Ollama Modelfile

**使用前请确保：**
- Colab 运行时设置为 GPU（T4）
- 已上传 `sft_train_p1.jsonl` 到 Colab

## 1. 环境准备

In [ ]:
# 检查 GPU
import torch
assert torch.cuda.is_available(), "请在 Colab 中设置为 GPU 运行时！"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# 安装依赖
!pip install -q transformers peft datasets accelerate bitsandbytes
print("依赖安装完成")

In [ ]:
# 上传 sft_train_p1.jsonl（或从 Google Drive 挂载）
# 方法 1: 直接上传
from google.colab import files
uploaded = files.upload()  # 选择 sft_train_p1.jsonl
print(f"已上传: {list(uploaded.keys())}")

## 2. 加载模型和数据

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
DATA_PATH = "sft_train_p1.jsonl"
MAX_LENGTH = 512

SYSTEM_PROMPT = (
    "你是一位中医老师，擅长用通俗易懂的方式讲解中医经典知识。"
    "请根据提供的经典原文回答问题。"
    "引用经典原文时标注条文编号。解释方剂时列出完整组成。"
    "不提供具体诊疗建议。如果检索结果中没有相关信息，请如实说明。"
)

# 加载 tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, padding_side="right")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 加载模型 (fp16)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
)
model.config.use_cache = False
print(f"模型参数量: {model.num_parameters() / 1e6:.1f}M")

In [ ]:
# 加载和预处理 SFT 数据
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw = [json.loads(line) for line in f]

print(f"训练样本数: {len(raw)}")
from collections import Counter
cats = Counter(r.get("category", "unknown") for r in raw)
print("类别分布:")
for cat, cnt in cats.most_common():
    print(f"  {cat}: {cnt}")

# 转为对话格式
records = []
for item in raw:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": item["instruction"]},
        {"role": "assistant", "content": item["output"]},
    ]
    records.append({"messages": messages})

dataset = Dataset.from_list(records)

# Tokenize
def format_to_ids(example):
    messages = example["messages"]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    full_ids = tokenizer(text, truncation=True, max_length=MAX_LENGTH, padding=False)["input_ids"]
    
    prefix_messages = messages[:2]
    prefix_text = tokenizer.apply_chat_template(prefix_messages, tokenize=False, add_generation_prompt=True)
    prefix_ids = tokenizer(prefix_text, truncation=True, max_length=MAX_LENGTH, padding=False)["input_ids"]
    
    labels = [-100] * len(prefix_ids) + full_ids[len(prefix_ids):]
    labels = labels[:len(full_ids)]
    
    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}

dataset = dataset.map(format_to_ids, remove_columns=dataset.column_names, desc="Tokenizing")
print(f"预处理完成: {len(dataset)} 条")

## 3. LoRA 微调训练

In [ ]:
# 配置 LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 训练
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

training_args = TrainingArguments(
    output_dir="./output_lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    fp16=True,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=True,
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True, return_tensors="pt")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)

print("开始训练...")
train_result = trainer.train()
print(f"\n训练完成！最终 loss: {train_result.training_loss:.4f}")

## 4. 合并 + 保存

In [ ]:
# 保存 LoRA adapter
model.save_pretrained("./output_lora")
tokenizer.save_pretrained("./output_lora")
print("LoRA adapter 已保存")

# 合并 adapter 到基座模型
print("合并 adapter...")
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./output_merged", safe_serialization=True, max_shard_size="2GB")
tokenizer.save_pretrained("./output_merged")
print("合并模型已保存到 ./output_merged/")

In [ ]:
# 快速验证微调效果
merged_model = merged_model.cuda()
test_questions = [
    "伤寒论第1条原文是什么？",
    "桂枝汤的组成是什么？",
    "什么是太阳病？",
]

for q in test_questions:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": q},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = merged_model.generate(**inputs, max_new_tokens=200, temperature=0.3, do_sample=True)
    response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"Q: {q}")
    print(f"A: {response}")
    print()

## 5. 转为 GGUF + 量化

In [ ]:
# 克隆 llama.cpp
!git clone https://github.com/ggerganov/llama.cpp.git
!pip install -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
print("llama.cpp 准备完成")

In [ ]:
# 转换为 GGUF (F16)
!python llama.cpp/convert_hf_to_gguf.py ./output_merged --outfile qwen25-15b-tcm-f16.gguf --outtype f16

import os
f16_size = os.path.getsize("qwen25-15b-tcm-f16.gguf") / 1024**3
print(f"F16 GGUF: {f16_size:.2f} GB")

In [ ]:
# 编译 llama-quantize 并量化为 Q4_K_M
!cd llama.cpp && make llama-quantize
!./llama.cpp/llama-quantize qwen25-15b-tcm-f16.gguf qwen25-15b-tcm-q4_k_m.gguf q4_k_m

q4_size = os.path.getsize("qwen25-15b-tcm-q4_k_m.gguf") / 1024**3
print(f"Q4_K_M GGUF: {q4_size:.2f} GB")

In [ ]:
# 生成 Modelfile
modelfile_content = '''FROM ./qwen25-15b-tcm-q4_k_m.gguf

TEMPLATE """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ .Response }}<|im_end|>
"""

SYSTEM """你是一位中医老师，擅长用通俗易懂的方式讲解中医经典知识。请根据提供的经典原文回答问题。引用经典原文时标注条文编号。解释方剂时列出完整组成。不提供具体诊疗建议。如果检索结果中没有相关信息，请如实说明。"""

PARAMETER stop "<|im_start|>"
PARAMETER stop "<|im_end|>"
PARAMETER temperature 0.7
PARAMETER top_p 0.8
PARAMETER top_k 20
'''

with open("Modelfile", "w", encoding="utf-8") as f:
    f.write(modelfile_content)
print("Modelfile 已生成")

## 6. 下载文件

下载以下两个文件到本地：
- `qwen25-15b-tcm-q4_k_m.gguf` (~1GB)
- `Modelfile`

In [ ]:
from google.colab import files
files.download("qwen25-15b-tcm-q4_k_m.gguf")
files.download("Modelfile")

## 7. 本地部署（下载完成后在本地执行）

```bash
# 将两个文件放在同一目录，然后：
ollama create qwen25-15b-tcm -f Modelfile

# 验证
ollama run qwen25-15b-tcm "什么是太阳病"
```

然后在项目中更新模型名：
```python
# src/rag/pipeline.py 中
pipeline = RAGPipeline(model='qwen25-15b-tcm')
```